In [1]:
import numpy as np
import torch
import itertools
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader,Dataset
import math
import random
from functools import partial
import argparse
from tqdm import tqdm
from torch.nn.functional import normalize
import scipy.sparse as sp
from numpy.linalg import inv
import copy
from torch.optim.lr_scheduler import _LRScheduler
import warnings
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module
torch.cuda.empty_cache()
device=torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
warnings.filterwarnings("ignore")
torch.cuda.is_available()

True

In [ ]:
import numpy as np
import torch
import itertools
def calculate_sim(aa,bb):
    s1=aa.shape[0]
    ll=torch.eye(s1)
    m2=bb*aa[:,None,:]
    m1=aa[:,:,None]
    for x,y in itertools.permutations(torch.linspace(0,s1-1,s1,dtype=torch.long),2):
        x,y=x.item(),y.item()
        m=m1[x,:,:]*m2[y,:,:]
        if aa[x].sum()+aa[y].sum()==0:
            ll[x,y]=0
        else:
            ll[x,y]=(m.max(dim=0,keepdim=True)[0].sum()+m.max(dim=1,keepdim=True)[0].sum())/(aa[x].sum()+aa[y].sum())
    return ll
def load_data():  # circ 834  dis 138  mi 555
    cd=np.load("./data_circ/circRNA_disease.npy")           
    dd=np.load("./data_circ/disease_disease.npy")      
    dm=np.load("./data_circ/disease_miRNA.npy")                        
    cm=np.load("./data_circ/circRNA_miRNA.npy")
    return torch.tensor(cd),torch.tensor(dd),torch.tensor(dm),torch.tensor(cm)
def split_dataset(al,dd,n,negr):#5 cross
    tri,tei,cda,cc=[],[],[],[]
    rand_index=torch.randperm(al.sum().long().item())
    ps=torch.argwhere(al==1).index_select(0,rand_index).T
    ns=torch.argwhere(al==0)
    ns=ns.index_select(0,torch.randperm(ns.shape[0])).T
    sf=int(ps.shape[1]/n)
    for i in range(n):
        ptrn=torch.cat([ps[:,:(i*sf)],ps[:,((i+1)*sf):(n*sf)]],dim=1)
        ntrn=torch.cat([ns[:,:(i*sf*negr)],ns[:,((i+1)*sf*negr):(n*sf*negr)]],dim=1)
        trn=torch.cat([ptrn,ntrn],dim=1)
        ten=torch.cat([ps[:,(i*sf):((i+1)*sf)],ns[:,(n*sf*negr):]],dim=1)
        tri.append(trn)
        tei.append(ten)
        cdt=al.clone()
        cdt[ps[0,(i*sf):((i+1)*sf)],ps[1,(i*sf):((i+1)*sf)]]=0
        cda.append(cdt)
        cc.append(calculate_sim(cdt,dd))
    return tri,tei,cda,cc
def cfm(cc,cd,dd,md,cm,mm): 
    r1=torch.cat([cc,cd,cm],dim=1)
    r2=torch.cat([cd.T,dd,md.T],dim=1)
    r3=torch.cat([cm.T,md,mm],dim=1)
    fea=torch.cat([r1,r2,r3],dim=0)
    return fea
n=5
neg_ratio=1
cd,dd,dm,cm=load_data()
tri,tei,cda,cc=split_dataset(cd,dd,n,neg_ratio)
mm=calculate_sim(dm.T,dd)
feas=[]
for i in range(n):
    fea=cfm(cc[i],cda[i],dd,dm.T,cm,mm)
    feas.append(fea)
torch.save([n,cd,feas,tri,tei],'./data_circ/circ_CNN.pth')

# process_data

In [5]:
p=None
feature_cd = torch.load('./data_circ/dataset/fea_cd')  #torch.Size([973, 1527])
adj_cd = torch.load('./data_circ/dataset/adj_cd')  #torch.Size([972, 972])

k1_cc = torch.tensor(20)    #原始图采样节点数
k1_dd = torch.tensor(20)
k1_cd = torch.tensor(4)
k1_dc = torch.tensor(4)

cc_shape=834
dd_shape=138

power_adj_list_cd5=copy.deepcopy(adj_cd)
for i in range(5):
    power_adj_list_cd5[i]=[power_adj_list_cd5[i]]
    for m in range(2):
        power_adj_list_cd5[i].append(power_adj_list_cd5[i][0]*power_adj_list_cd5[i][m])      #(5,3,972,972)

#Sampling heuristics: 0,1,2
eigen_adj_cd5,eigen_adj1_cd5,eigen_adj2_cd5=[],[],[]
for i in range(5):
    one = power_adj_list_cd5[i][0]  #一阶邻接矩阵
    two = power_adj_list_cd5[i][1]    #二阶邻接矩阵
    three = power_adj_list_cd5[i][2]    #二阶邻接矩阵
    eigen_adj_cd5.append(one)
    eigen_adj1_cd5.append(two)
    eigen_adj2_cd5.append(three)

eigen_adj_cd5_copy1 = copy.deepcopy(eigen_adj_cd5)
eigen_adj_cd5_copy2 = copy.deepcopy(eigen_adj_cd5)
eigen_adj_cd5_copy3 = copy.deepcopy(eigen_adj_cd5)
eigen_adj_cd5_copy4 = copy.deepcopy(eigen_adj_cd5)

eigen_adj1_cd5_copy1 = copy.deepcopy(eigen_adj1_cd5)
eigen_adj1_cd5_copy2 = copy.deepcopy(eigen_adj1_cd5)
eigen_adj1_cd5_copy3 = copy.deepcopy(eigen_adj1_cd5)
eigen_adj1_cd5_copy4 = copy.deepcopy(eigen_adj1_cd5)

eigen_adj2_cd5_copy1 = copy.deepcopy(eigen_adj2_cd5)
eigen_adj2_cd5_copy2 = copy.deepcopy(eigen_adj2_cd5)
eigen_adj2_cd5_copy3 = copy.deepcopy(eigen_adj2_cd5)
eigen_adj2_cd5_copy4 = copy.deepcopy(eigen_adj2_cd5)


torch.Size([973, 973])


In [2]:
#保存的是大矩阵里的下标
data_list_cc = []          #空列表存储子图样本和相关数据，五折
data_list_dd = []          #空列表存储子图样本和相关数据，五折


for i in range(5):
    sub_data_list_cc = []      #存储当前节点子图样本和相关数据
    for id in range(cc_shape):       #遍历原始图所有节点
        s = eigen_adj_cd5_copy1[i][id]
        s[id] = 0
        s[834:] = 0
        #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
        if p is not None:           #p是各种策略权重
            s1 = eigen_adj1_cd5_copy1[i][id]
            s2 = eigen_adj2_cd5_copy1[i][id]
            s1[id] = 0
            s2[id] = 0
            s1[834:] = 0
            s2[834:] = 0
            #得到最终采样概率
            s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
        # sample_num1 = np.minimum(k1_cc, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
        print(s.shape)
        sample_num1 = torch.min(k1_cc, (s > 0).sum())
        if sample_num1 > 0:
            #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
            #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
            sample_index1 = s.argsort()[-(sample_num1):].cpu()
        else:
            #不进行采样
            sample_index1 = np.array([], dtype=int)
        #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
        #采样够14个节点就行，不够的话用top邻居补上
        node_feature_id = torch.cat([torch.tensor([id, ]), torch.tensor(sample_index1, dtype=int),
                                torch.ones(k1_cc-sample_num1, dtype=int)*1527])
        #如果这个条件为真（即相等），那么程序会继续执行。
        # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。    
        assert len(node_feature_id) == k1_cc+1
        sub_data_list_cc.append(node_feature_id) 
        

    #将所有子图样本添加到 data_list 中，构建了数据集的一部分。
    sub_data_list_cd = []      #存储当前节点子图样本和相关数据
    for id in range(cc_shape):       #遍历原始图所有节点
        s = eigen_adj_cd5_copy2[i][id]
        s[:834]=0
        #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
        if p is not None:           #p是各种策略权重
            s1 = eigen_adj1_cd5_copy2[i][id]
            s2 = eigen_adj2_cd5_copy2[i][id]
            s1[:834]=0
            s2[:834]=0
            #得到最终采样概率
            s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
        # sample_num1 = np.minimum(k1_cd, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
        sample_num1 = torch.min(k1_cd, (s > 0).sum())
        if sample_num1 > 0:
            #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
            #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
            sample_index1 = s.argsort()[-(sample_num1):].cpu()
        else:
            #不进行采样
            sample_index1 = np.array([], dtype=int)
        #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
        #采样够14个节点就行，不够的话用top邻居补上
        node_feature_id = torch.cat([torch.tensor(sample_index1, dtype=int),
                                torch.ones(k1_cd-sample_num1, dtype=int)*1527])
        #如果这个条件为真（即相等），那么程序会继续执行。
        # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。
        assert len(node_feature_id) == k1_cd
        sub_data_list_cd.append(node_feature_id)  
    result_list = [torch.cat((tensor1, tensor2), dim=0) for tensor1, tensor2 in zip(sub_data_list_cc, sub_data_list_cd)]
    data_list_cc.append(result_list)
    
    
    
for i in range(5):
    sub_data_list_dd = []      #存储当前节点子图样本和相关数据
    for id in range(dd_shape):       #遍历原始图所有节点
        s = eigen_adj_cd5_copy3[0][id+834]
        s[id+834] = 0
        s[:834]=0
        #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
        if p is not None:           #p是各种策略权重
            s1 = eigen_adj1_cd5_copy3[0][id+834]
            s2 = eigen_adj2_cd5_copy3[0][id+834]
            s1[id+834] = 0
            s1[:834]=0
            s2[id+834] = 0
            s2[:834]=0
            #得到最终采样概率
            s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
        # sample_num1 = np.minimum(k1_dd, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
        sample_num1 = torch.min(k1_dd, (s > 0).sum())
        if sample_num1 > 0:
            #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
            #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
            sample_index1 = s.argsort()[-(sample_num1):].cpu()
        else:
            #不进行采样
            sample_index1 = np.array([], dtype=int)
        #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
        #采样够14个节点就行，不够的话用top邻居补上
        node_feature_id = torch.cat([torch.tensor([id+834, ]), torch.tensor(sample_index1, dtype=int),
                                torch.ones(k1_dd-sample_num1, dtype=int)*1527])
        #如果这个条件为真（即相等），那么程序会继续执行。
        # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。
        assert len(node_feature_id) == k1_dd+1
        sub_data_list_dd.append(node_feature_id)    
    #将所有子图样本添加到 data_list 中，构建了数据集的一部分。
    
     #将所有子图样本添加到 data_list 中，构建了数据集的一部分。
    sub_data_list_dc = []      #存储当前节点子图样本和相关数据
    for id in range(dd_shape):       #遍历原始图所有节点
        s = eigen_adj_cd5_copy4[i][id+834]
        s[834:] = 0
        #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
        if p is not None:           #p是各种策略权重
            s1 = eigen_adj1_cd5_copy4[i][id+834]
            s2 = eigen_adj2_cd5_copy4[i][id+834]
            s1[:834] = 0
            s2[:834] = 0
            #得到最终采样概率
            s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
        # sample_num1 = np.minimum(k1_dc, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
        sample_num1 = torch.min(k1_dc, (s > 0).sum())
        if sample_num1 > 0:
            #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
            #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
            sample_index1 = s.argsort()[-(sample_num1):].cpu()
        else:
            #不进行采样
            sample_index1 = np.array([], dtype=int)
        #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
        #采样够14个节点就行，不够的话用top邻居补上
        node_feature_id = torch.cat([torch.tensor(sample_index1, dtype=int),
                                torch.ones(k1_dc-sample_num1, dtype=int)*1527])
        #如果这个条件为真（即相等），那么程序会继续执行。
        # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。
        assert len(node_feature_id) == k1_dc
        sub_data_list_dc.append(node_feature_id)  
    result_list = [torch.cat((tensor1, tensor2), dim=0) for tensor1, tensor2 in zip(sub_data_list_dd, sub_data_list_dc)]
    data_list_dd.append(result_list)

torch.save(data_list_cc,'./data_circ/dataset/data_cc.pt')
torch.save(data_list_dd,'./data_circ/dataset/data_dd.pt')


KeyboardInterrupt



# sample

In [6]:
def node_sampling(p=None):
    feature_cd = torch.load('./data_circ/dataset/fea_cd')  #torch.Size([973, 1527])
    adj_cd = torch.load('./data_circ/dataset/adj_cd')  #torch.Size([972, 972])

    c = 0.15  #ppr中的c
    k1_cc = torch.tensor(20)    #原始图采样节点数
    k1_dd = torch.tensor(20)
    k1_cd = torch.tensor(4)
    k1_dc = torch.tensor(4)
    k2 = 0      #粗图采样节点数
    cc_shape=834
    dd_shape=138

    power_adj_list_cd5=copy.deepcopy(adj_cd)
    for i in range(5):
        power_adj_list_cd5[i]=[power_adj_list_cd5[i]]
        for m in range(2):
            power_adj_list_cd5[i].append(power_adj_list_cd5[i][0]*power_adj_list_cd5[i][m])      #(5,3,972,972)

    #Sampling heuristics: 0,1,2
    eigen_adj_cd5,eigen_adj1_cd5,eigen_adj2_cd5=[],[],[]
    for i in range(5):
        one = power_adj_list_cd5[i][0]  #一阶邻接矩阵
        two = power_adj_list_cd5[i][1]    #二阶邻接矩阵
        three = power_adj_list_cd5[i][2]    #二阶邻接矩阵
        eigen_adj_cd5.append(one)
        eigen_adj1_cd5.append(two)
        eigen_adj2_cd5.append(three)

    eigen_adj_cd5_copy1 = copy.deepcopy(eigen_adj_cd5)
    eigen_adj_cd5_copy2 = copy.deepcopy(eigen_adj_cd5)
    eigen_adj_cd5_copy3 = copy.deepcopy(eigen_adj_cd5)
    eigen_adj_cd5_copy4 = copy.deepcopy(eigen_adj_cd5)

    eigen_adj1_cd5_copy1 = copy.deepcopy(eigen_adj1_cd5)
    eigen_adj1_cd5_copy2 = copy.deepcopy(eigen_adj1_cd5)
    eigen_adj1_cd5_copy3 = copy.deepcopy(eigen_adj1_cd5)
    eigen_adj1_cd5_copy4 = copy.deepcopy(eigen_adj1_cd5)

    eigen_adj2_cd5_copy1 = copy.deepcopy(eigen_adj2_cd5)
    eigen_adj2_cd5_copy2 = copy.deepcopy(eigen_adj2_cd5)
    eigen_adj2_cd5_copy3 = copy.deepcopy(eigen_adj2_cd5)
    eigen_adj2_cd5_copy4 = copy.deepcopy(eigen_adj2_cd5)

    
    #保存的是大矩阵里的下标
    data_list_cc = []          #空列表存储子图样本和相关数据，五折
    data_list_dd = []          #空列表存储子图样本和相关数据，五折


    for i in range(5):
        sub_data_list_cc = []      #存储当前节点子图样本和相关数据
        for id in range(cc_shape):       #遍历原始图所有节点
            s = eigen_adj_cd5_copy1[i][id]
            s[id] = 0
            s[834:] = 0
            #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
            if p is not None:           #p是各种策略权重
                s1 = eigen_adj1_cd5_copy1[i][id]
                s2 = eigen_adj2_cd5_copy1[i][id]
                s1[id] = 0
                s2[id] = 0
                s1[834:] = 0
                s2[834:] = 0
                #得到最终采样概率
                s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
            # sample_num1 = np.minimum(k1_cc, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
            sample_num1 = torch.min(k1_cc, (s > 0).sum())
            if sample_num1 > 0:
                #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
                #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
                sample_index1 = s.argsort()[-(sample_num1):].cpu()
            else:
                #不进行采样
                sample_index1 = np.array([], dtype=int)
            #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
            #采样够14个节点就行，不够的话用top邻居补上
            node_feature_id = torch.cat([torch.tensor([id, ]), torch.tensor(sample_index1, dtype=int),
                                    torch.ones(k1_cc-sample_num1, dtype=int)*1527])
            #如果这个条件为真（即相等），那么程序会继续执行。
            # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。    
            assert len(node_feature_id) == k1_cc+1
            sub_data_list_cc.append(node_feature_id) 


        #将所有子图样本添加到 data_list 中，构建了数据集的一部分。
        sub_data_list_cd = []      #存储当前节点子图样本和相关数据
        for id in range(cc_shape):       #遍历原始图所有节点
            s = eigen_adj_cd5_copy2[i][id]
            s[:834]=0
            #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
            if p is not None:           #p是各种策略权重
                s1 = eigen_adj1_cd5_copy2[i][id]
                s2 = eigen_adj2_cd5_copy2[i][id]
                s1[:834]=0
                s2[:834]=0
                #得到最终采样概率
                s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
            # sample_num1 = np.minimum(k1_cd, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
            sample_num1 = torch.min(k1_cd, (s > 0).sum())
            if sample_num1 > 0:
                #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
                #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
                sample_index1 = s.argsort()[-(sample_num1):].cpu()
            else:
                #不进行采样
                sample_index1 = np.array([], dtype=int)
            #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
            #采样够14个节点就行，不够的话用top邻居补上
            node_feature_id = torch.cat([torch.tensor(sample_index1, dtype=int),
                                    torch.ones(k1_cd-sample_num1, dtype=int)*1527])
            #如果这个条件为真（即相等），那么程序会继续执行。
            # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。
            assert len(node_feature_id) == k1_cd
            sub_data_list_cd.append(node_feature_id)  
        result_list = [torch.cat((tensor1, tensor2), dim=0) for tensor1, tensor2 in zip(sub_data_list_cc, sub_data_list_cd)]
        data_list_cc.append(result_list)



    for i in range(5):
        sub_data_list_dd = []      #存储当前节点子图样本和相关数据
        for id in range(dd_shape):       #遍历原始图所有节点
            s = eigen_adj_cd5_copy3[0][id+834]
            s[id+834] = 0
            s[:834]=0
            #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
            if p is not None:           #p是各种策略权重
                s1 = eigen_adj1_cd5_copy3[0][id+834]
                s2 = eigen_adj2_cd5_copy3[0][id+834]
                s1[id+834] = 0
                s1[:834]=0
                s2[id+834] = 0
                s2[:834]=0
                #得到最终采样概率
                s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
            # sample_num1 = np.minimum(k1_dd, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
            sample_num1 = torch.min(k1_dd, (s > 0).sum())
            if sample_num1 > 0:
                #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
                #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
                sample_index1 = s.argsort()[-(sample_num1):].cpu()
            else:
                #不进行采样
                sample_index1 = np.array([], dtype=int)
            #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
            #采样够14个节点就行，不够的话用top邻居补上
            node_feature_id = torch.cat([torch.tensor([id+834, ]), torch.tensor(sample_index1, dtype=int),
                                    torch.ones(k1_dd-sample_num1, dtype=int)*1527])
            #如果这个条件为真（即相等），那么程序会继续执行。
            # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。
            assert len(node_feature_id) == k1_dd+1
            sub_data_list_dd.append(node_feature_id)    
        #将所有子图样本添加到 data_list 中，构建了数据集的一部分。

         #将所有子图样本添加到 data_list 中，构建了数据集的一部分。
        sub_data_list_dc = []      #存储当前节点子图样本和相关数据
        for id in range(dd_shape):       #遍历原始图所有节点
            s = eigen_adj_cd5_copy4[i][id+834]
            s[834:] = 0
            #即第一次采样仅使用一阶邻居进行采样，后续采用到自适应采样策略
            if p is not None:           #p是各种策略权重
                s1 = eigen_adj1_cd5_copy4[i][id+834]
                s2 = eigen_adj2_cd5_copy4[i][id+834]
                s1[:834] = 0
                s2[:834] = 0
                #得到最终采样概率
                s = p[0]*s/(s.sum()+1e-5) + p[1]*s1/(s1.sum()+1e-5) + p[2]*s2/(s2.sum()+1e-5)
            # sample_num1 = np.minimum(k1_dc, (s > 0).sum())     #原始图采样节点数取14和一阶邻居正例最小值
            sample_num1 = torch.min(k1_dc, (s > 0).sum())
            if sample_num1 > 0:
                #随机选择sample_num1个采样节点，不允许重复，且概率更大更有可能被选中
                #sample_index1 = np.random.choice(a=np.arange(fea[i].shape[0]), size=sample_num1, replace=False, p=s/s.sum())
                sample_index1 = s.argsort()[-(sample_num1):].cpu()
            else:
                #不进行采样
                sample_index1 = np.array([], dtype=int)
            #创建一个包含当前节点、sample_index1 和 top_neighbor_index 的列表，这些将成为子图的节点特征。
            #采样够14个节点就行，不够的话用top邻居补上
            node_feature_id = torch.cat([torch.tensor(sample_index1, dtype=int),
                                    torch.ones(k1_dc-sample_num1, dtype=int)*1527])
            #如果这个条件为真（即相等），那么程序会继续执行。
            # 如果条件为假（即不相等），则会引发 AssertionError，程序将停止执行。
            assert len(node_feature_id) == k1_dc
            sub_data_list_dc.append(node_feature_id)  
        result_list = [torch.cat((tensor1, tensor2), dim=0) for tensor1, tensor2 in zip(sub_data_list_dd, sub_data_list_dc)]
        data_list_dd.append(result_list)

    return data_list_cc,data_list_dd

# p=[0.25,0.25,0.25]
# data_list_cc, data_list_dd = node_sampling(p)
# print(len(data_list_cc))#(5,834,25)
# print(len(data_list_dd))#(5,138,25)


# model

In [3]:
class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features):
        super(GraphConvolution, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight)

    def forward(self, input, adj):
        support = torch.mm(input, self.weight)
        output = torch.mm(adj, support)
        return output

class SelfAttention(nn.Module):
    def __init__(self, in_features, out_features, heads=1):
        super(SelfAttention, self).__init__()
        self.heads = heads
        self.query = nn.Linear(in_features, out_features)
        self.key = nn.Linear(in_features, out_features)
        self.value = nn.Linear(in_features, out_features)

    def forward(self, x):
        batch_size = 1
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        Q = Q.view(batch_size, -1, self.heads, Q.size(-1) // self.heads)
        K = K.view(batch_size, -1, self.heads, K.size(-1) // self.heads)
        V = V.view(batch_size, -1, self.heads, V.size(-1) // self.heads)

        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        V = V.permute(0, 2, 1, 3)

        attention = torch.matmul(Q, K.transpose(-2, -1)) / (Q.size(-1) ** 0.5)
        attention = torch.softmax(attention, dim=-1)

        out = torch.matmul(attention, V)
        out = out.permute(0, 2, 1, 3).contiguous()
        out = out.view( -1, self.heads * (Q.size(-1)))

        return out

class MLP(nn.Module):
    def __init__(self, in_features, out_features):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(in_features, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, out_features)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    

class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha,concat=True):
        super(GraphAttentionLayer, self).__init__()
        self.dropout = dropout
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.concat = concat

        
        self.Wcc = nn.Linear(in_features, out_features)
        self.Wcd = nn.Linear(in_features, out_features)
        self.Wdd = nn.Linear(in_features, out_features)
        self.Wdc = nn.Linear(in_features, out_features)
        self.a = nn.Linear(out_features * 2, 1)
        nn.init.xavier_normal_(self.Wcc.weight)
        nn.init.xavier_normal_(self.Wcd.weight)
        nn.init.xavier_normal_(self.Wdd.weight)
        nn.init.xavier_normal_(self.Wdc.weight)
        nn.init.xavier_normal_(self.a.weight)
        
        self.leakyrelu = nn.LeakyReLU(0.2)
        
    def forward(self, h,start,get_score=False):
        h_neighbors=h[:,0:1,:].repeat(1,h.shape[1],1)
        #cstart
        if start ==0:
            h_neighbors_1=self.Wcc(h_neighbors[:,:20,:]) 
            h_neighbors_2=self.Wcd(h_neighbors[:,20:,:]) 
            h_neighbors=torch.cat([h_neighbors_1,h_neighbors_2],dim=1)
            h_1=self.Wcc(h[:,:20,:])
            h_2=self.Wcd(h[:,20:,:])
            h=torch.cat([h_1,h_2],dim=1)
            combined=torch.cat([h_neighbors,h],dim=2)
            e=self.leakyrelu(self.a(combined))
        if start ==1:
            h_neighbors_1=self.Wdd(h_neighbors[:,:20,:]) 
            h_neighbors_2=self.Wdc(h_neighbors[:,20:,:]) 
            h_neighbors=torch.cat([h_neighbors_1,h_neighbors_2],dim=1)
            h_1=self.Wdd(h[:,:20,:])
            h_2=self.Wdc(h[:,20:,:])
            h=torch.cat([h_1,h_2],dim=1)
            combined=torch.cat([h_neighbors,h],dim=2)
            e=self.leakyrelu(self.a(combined))
        e=torch.transpose(e,1,2)
        attention = F.softmax(e, dim=2)
        if get_score:
            score = attention.squeeze()
        h_prime = torch.matmul(attention, h)
        h_prime=h_prime.squeeze()
        if get_score:
            return score
        else:
            if self.concat:
                return F.leaky_relu(h_prime)
            else:
                return h_prime

    def __repr__(self):
        return self.__class__.__name__ + ' (' + str(self.in_features) + ' -> ' + str(self.out_features) + ')'

    

class GAT(nn.Module):
    def __init__(self, nfeat, nhid, dropout, alpha, nheads):
        super(GAT, self).__init__()
        self.dropout = dropout
        
        self.attentions1 = [GraphAttentionLayer(nfeat, nhid,dropout=dropout, alpha=alpha,concat=True) for _ in range(nheads)]
        for i, attention in enumerate(self.attentions1):
            self.add_module('attention1_{}'.format(i), attention)
        self.out_att1 = nn.Linear(nhid * nheads, nhid)  # 输出层
        self.c1=nn.Conv2d(1,32,kernel_size=(2,4),stride=1,padding=0)
        self.s1=nn.MaxPool2d(kernel_size=(1,6))
        self.c2=nn.Conv2d(32,64,kernel_size=(1,4),stride=1,padding=0)
        self.s2=nn.MaxPool2d(kernel_size=(1,10))
        self.l1=nn.Sequential(nn.Linear(76*64,1024),nn.LeakyReLU(),nn.Dropout(0.5),nn.Linear(1024,512),nn.LeakyReLU(),nn.Dropout(0.5))
        self.l2=nn.Sequential(nn.Linear(512,256),nn.LeakyReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        self.leakyrelu=nn.LeakyReLU()
        self.g1 = nn.Linear(1527, 1527)  
        self.g2 = nn.Linear(1527, 1527)  
        self.g3 = nn.Linear(1527, 1527) 
        self.g4 = nn.Linear(1527, 1527) 
        self.bias = nn.Parameter(torch.zeros(1527))
        self.gc1 = GraphConvolution(nfeat, nhid)
        self.gc2 = GraphConvolution(nfeat, nhid)
        self.attention2 = SelfAttention(nfeat, nhid, heads=3)
        self.mlp = MLP(nhid, 128)
        self.reset_para()
        
    def reset_para(self):
        nn.init.xavier_normal_(self.c1.weight)
        nn.init.xavier_normal_(self.c2.weight)
        nn.init.xavier_normal_(self.g1.weight)
        for mode in self.l1:
            if isinstance(mode,nn.Linear):
                nn.init.xavier_normal_(mode.weight,gain= nn.init.calculate_gain('relu'))
        for mode in self.l2:
            if isinstance(mode,nn.Linear):
                nn.init.xavier_normal_(mode.weight,gain= nn.init.calculate_gain('relu'))

    def forward(self, x1,x2,fea,adj,fea_ori,adj_ori,data_list_cc,data_list_dd,X_new,get_score=False):
       
        fea_ori_first=fea_ori
        
        selected_row1 = data_list_cc[x1]
        selected_row2 = data_list_dd[x2]
        selected_feature1=fea[selected_row1]     #(16,20,1527)
        selected_feature2=fea[selected_row2]     #(16,20,1527)
        selected_feature1 = F.dropout(selected_feature1, self.dropout, training=self.training)
        selected_feature2 = F.dropout(selected_feature2, self.dropout, training=self.training)
        
        relation_cstart=0
        relation_dstart=1
        
        if get_score:
            score1 = [att(selected_feature1,relation_cstart,get_score=True) for att in self.attentions1]
            score1=torch.stack(score1,dim=0)
            score1=score1.mean(dim=0)
            score2 = [att(selected_feature2,relation_dstart,get_score=True) for att in self.attentions1]
            score2=torch.stack(score2,dim=0)
            score2=score2.mean(dim=0)
            return score1,score2
        
        output1 = torch.cat([att(selected_feature1,relation_cstart) for att in self.attentions1], dim=1)
        output1 = self.out_att1(output1)  # 最终的输出层
        output2 = torch.cat([att(selected_feature2,relation_dstart) for att in self.attentions1], dim=1)
        output2 = self.out_att1(output2)  # 最终的输出层
        output1 = F.dropout(output1, self.dropout, training=self.training)
        output2 = F.dropout(output2, self.dropout, training=self.training)
        
        
        
        ori1=fea_ori[x1]
        ori2=fea_ori[x2+834]
        
        
        # 计算 S
        
        Z = self.attention2(fea_ori)
        S = torch.matmul(Z, Z.transpose(0, 1))

        

        # 计算 D^{-1} for A ⊙ S
        A_S = torch.mul(adj_ori, S)
        degree = A_S.sum(0)
        D_inv = torch.diag(1.0/degree)
        
        # 计算 D^{-1} (A ⊙ S) 
        DA = torch.mm(D_inv, A_S)
        
#         # 计算第二层的输出

        feak = F.relu(self.gc1(fea_ori, DA))
        feak2 = F.relu(self.gc2(feak, DA))
        
        # 计算最终的输出
        # Z = F.relu(torch.matmul(Z_ORI, self.W1) + self.gc2(fea, DA))
        
        
        con1=feak2[x1]
        con2=feak2[x2+834]
        

        gate1=F.sigmoid(self.g1(output1)+self.g2(con1)+self.bias)
        output1=gate1*output1
        con1=(1-gate1)*con1
        

        gate2=F.sigmoid(self.g3(output2)+self.g4(con2)+self.bias)
        output2=gate2*output2
        con2=(1-gate2)*con2
        

        
        
        output1=torch.cat([ori1,output1,con1],dim=1)
        output2=torch.cat([ori2,output2,con2],dim=1)
        # y=torch.cat([output1,output2],dim=1) 
        output1 = output1[:,None,None, :]
        output2 = output2[:,None,None, :]
        y=torch.cat([output1,output2],dim=2) 
        y=self.s1(self.leakyrelu(self.c1(y)))  
        y=self.s2(self.leakyrelu(self.c2(y))) 
        y=y.reshape(y.shape[0],-1)
        y=self.l1(y)
        y=self.l2(y)
        return y
    

    

# net = GAT(
#         nfeat=1527,
#         nhid=1527,
#         dropout=0.3,
#         nheads=8,
#         alpha=0.2
#     ).to(device)

# x1=torch.tensor([656,642,424,662,697,819,478,36,591,509,806,699,723,670,511,679],dtype=torch.long).to(device)
# x2=torch.tensor([88,2,92,65,110,77,108,12,107,118,136,109,128,103,117,69],dtype=torch.long).to(device)
# adj_cd = torch.load('./data_circ/dataset/adj_cd')
# feature_cd = torch.load('./data_circ/dataset/fea_cd')
# feature_cd= torch.stack(feature_cd).to(device)
# adj_cd= torch.stack(adj_cd).to(device)
# data_list_cc = torch.load('./data_circ/dataset/data_cc.pt')
# data_list_dd = torch.load('./data_circ/dataset/data_dd.pt')
# data_list_cc = torch.stack([torch.stack(inner_list) for inner_list in data_list_cc]).to(device)
# data_list_dd = torch.stack([torch.stack(inner_list) for inner_list in data_list_dd]).to(device)
# data_list_cc[data_list_cc == 1527] = 972
# data_list_dd[data_list_dd == 1527] = 972


# adj_ori = adj_cd[:, :972, :972]     #已经归一化了
# feature_ori = feature_cd[:, :972, :]
# X_new = torch.matmul(adj_ori, feature_ori)


# one,two=net(x1,x2,feature_cd[0],adj_cd[0],feature_ori[0],adj_ori[0],data_list_cc[0],data_list_dd[0],X_new,get_score=True)#(32,2)
# print(one.shape)
# print(two.shape)
# net(x1,x2,feature_cd[0],adj_cd[0],feature_ori[0],adj_ori[0],data_list_cc[0],data_list_dd[0],X_new[0]).shape#(32,2)

 # get_reward

In [8]:
#计算四种策略的奖励
def get_reward(i,args, model, device, loader, p,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new):
    
    ori_data_list_cc=data_list_cc
    ori_data_list_dd=data_list_dd
    
    
    power_adj_list_cd5=copy.deepcopy(adj)
    power_adj_list_cd5=[power_adj_list_cd5]
    for m in range(2):
        power_adj_list_cd5.append(power_adj_list_cd5[0]*power_adj_list_cd5[m])      #(3,972,972)
    
    #归一化邻接矩阵，一跳邻居
    eigen_adj_cd5 = power_adj_list_cd5[0]
    #保存二跳举例
    eigen_adj1_cd5 = power_adj_list_cd5[1]
    #这个 PageRank 矩阵可以用来估计每个节点在网络中的重要性或排名
    eigen_adj2_cd5 = power_adj_list_cd5[2]

    r = [[], [], []]
    reward = np.zeros(3)
    model.eval()    #评估模式

    
    n_node = 25
    #返回迭代的索引 step 和对应的数据批次 batch
    for x1,x2,y in tqdm(loader, desc="Iteration"):
        x1,x2,y=x1.long().to(device),x2.long().to(device),y.long().to(device)
        with torch.no_grad():
            #返回分数
            scores1,scores2 = model(x1,x2,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new,get_score=True)
            #scores 变量的列从第1列到第 n_node - 1 列（不包括第 n_node 列）进行切片。
            # 这可能是因为在计算中不考虑第一列或最后一列的数据。
        x1,x2=x1.to('cpu'),x2.to('cpu')
        scores1 = scores1[:, 1:n_node]   #(32,24)   取分数
        scores2 = scores2[:, 1:n_node]  #(32,24)
        
        ids1=data_list_cc[x1]        #(32,24)   取对应索引
        ids1 = ids1[:, 1:n_node]   #(32,24)
        
        ids2=data_list_dd[x2]
        ids2 = ids2[:, 1:n_node]
        
        data_list_cc=ori_data_list_cc
        data_list_dd=ori_data_list_dd
        

        
        ids1[ids1 == 1527] = 972
        ids2[ids2 == 1527] = 972
        
        
        #循环遍历模型输出的分数 scores
        for i,score in enumerate(scores1):
            id = x1[i]    #（取行）
            ids=ids1[i]     #（取索引）
            s = eigen_adj_cd5[id]#(1527,)
            s1 = eigen_adj1_cd5[id]
            s2 = eigen_adj2_cd5[id]
            s[id], s1[id], s2[id] = 0, 0, 0
            s = s/(s.sum()+1e-5)
            s1 = s1/(s1.sum()+1e-5)
            s2 = s2/(s2.sum()+1e-5)
            phi = p[0]*s + p[1]*s1 + p[2]*s2 + 1e-5
            r[0].append(torch.sum(score * s[ids] / phi[ids]))
            r[1].append(torch.sum(score * s1[ids] / phi[ids]))
            r[2].append(torch.sum(score * s2[ids] / phi[ids]))
            
            
        for i,score in enumerate(scores2):
            id = x2[i]+834
            ids=ids2[i]
            s = eigen_adj_cd5[id]#(1527,)
            s1 = eigen_adj1_cd5[id]
            s2 = eigen_adj2_cd5[id]
            s[id], s1[id], s2[id] = 0, 0, 0
            s = s/(s.sum()+1e-5)
            s1 = s1/(s1.sum()+1e-5)
            s2 = s2/(s2.sum()+1e-5)
            phi = p[0]*s + p[1]*s1 + p[2]*s2 +  1e-5
            r[0].append(torch.sum(score * s[ids] / phi[ids]))
            r[1].append(torch.sum(score * s1[ids] / phi[ids]))
            r[2].append(torch.sum(score * s2[ids] / phi[ids]))
    reward[0] = torch.mean(torch.cat([i.unsqueeze(0) for i in r[0]])).cpu().numpy()
    reward[1] = torch.mean(torch.cat([i.unsqueeze(0) for i in r[1]])).cpu().numpy()
    reward[2] = torch.mean(torch.cat([i.unsqueeze(0) for i in r[2]])).cpu().numpy()
    return reward

# trset=DataLoader(MyDataset(tri[i],cd),args.batch_size,shuffle=True)      #读训练数据，格式（32,x1,x2,label）
# teset=DataLoader(MyDataset(tei[i],cd),args.batch_size,shuffle=False)     #读测试数据
# data_list_cc=torch.load('./data_circ/dataset/data_cc.pt')
# data_list_dd=torch.load('./data_circ/dataset/data_dd.pt')
# feature=torch.load('./data_circ/dataset/fea_cd')
# feature= torch.stack(feature).to(device)
# adj=torch.load('./data_circ/dataset/adj_cd')
# adj= torch.stack(adj).to(device)
# data_list_cc = torch.stack([torch.stack(inner_list) for inner_list in data_list_cc]).to(device)
# data_list_dd = torch.stack([torch.stack(inner_list) for inner_list in data_list_dd]).to(device)
# data_list_cc[data_list_cc == 1527] = 972
# data_list_dd[data_list_dd == 1527] = 972
# adj_ori = adj_cd[:, :972, :972]     #已经归一化了
# feature_ori = feature_cd[:, :972, :]
# s_ax = torch.load('./data_circ/dataset/s_ax')  
# s_a2x = torch.load('./data_circ/dataset/s_a2x') 
# p=[0.25,0.25,0.25]
# r = get_reward(i,args, model, device, trset, p,feature_cd[i],adj_cd[i],feature_ori[i],adj_ori[i],data_list_cc[i],data_list_dd[i],s_ax[i],s_a2x[i])
# print('reward:', r)

# train

In [9]:
#模型，设别，数据加载器，优化器，学习率调度器
def train(args, model, device, loader, optimizer,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new,teset,cros,epoch ,epochs):
    isSave=0
    cost=nn.CrossEntropyLoss()
    running_loss = 0.0
    model.train()       #模型设置为训练模式(启用dropout或批量归一化)
#         迭代器，遍历loader中的每个批次，tqdm是一个用于在命令行中显示进度条的库
    for x1,x2,y in tqdm(loader, desc="Iteration"):
        x1,x2,y = x1.long().to(device),x2.long().to(device),y.long().to(device)    #批次移动到GPU
        pred = model(x1,x2,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new)         #前向传播
        loss = cost(pred, y)     #计算损失
        optimizer.zero_grad()               #梯度清零
        loss.backward()             #反向传播，计算梯度
        optimizer.step()            #更新参数，最小化损
        running_loss += loss.item()
    print(f"Loss: {running_loss}")
    if epoch==epochs:
        isSave=1
        tacc(args,model,loader,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new,0,isSave,cros)      #训练集准确率
        tacc(args,model,teset,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new,1,isSave,cros)        #测试集准确率
        torch.save(model.state_dict(), './best_network.pth')
        torch.cuda.empty_cache()

def tacc(args,model,tset,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new,string,s,cros):
    correct=0      #预测正确数
    total=0        #样本总数
    st={0:'train_acc',1:'test_acc'}
    predall,yall=torch.tensor([]).to(device),torch.tensor([]).to(device)     #存预测值和标签
    model.eval()      #测试模式，droupout无效
    for x1,x2,y in tset:
        x1,x2,y=x1.long().to(device),x2.long().to(device),y.long().to(device)
        pred=model(x1,x2,feature,adj,feature_ori,adj_ori,data_list_cc,data_list_dd,X_new).data     #得到预测值(32,2)
        if s==1:
            predall=torch.cat([predall,torch.as_tensor(pred)],dim=0)
            yall=torch.cat([yall,torch.as_tensor(y)])
        a=torch.max(pred,1)[1]#   pred按行取最大值的索引
        total+=y.size(0)    #总数相加
        correct+=(a==y).sum()     #预测对的值相加
    if string==1 and s==1:
        print('yes')
        torch.save((predall,yall),'./data_circ/GAT_topk4/circ_plt_%d'%cros)
    print(st[string]+str((correct/total).item()))


In [10]:
# # adj_cd = torch.load('./data_circ/dataset/adj_cd')
# # adj_cd= torch.stack(adj_cd).to(device)
# # print(adj_cd.shape)
# # print(adj_cd[0][0].sum())
# new_tensor_1 = torch.zeros((5, 972, 1527))
# new_tensor_1[:, :972, :1527] = feature[:, :972, :1527]
# print(new_tensor_1.shape)
# print(new_tensor_1[0][0].sum())
# tensor_list_1 = []
# for i in range(5):
#     channel_tensor = new_tensor_1[i, :, :]  # 获取第i个通道的张量
#     tensor_list_1.append(channel_tensor)
# torch.save(tensor_list_1,'./data_circ/dataset/fea_cd')

In [11]:
#用于处理自己的数据，输出坐标和cd矩阵，返回坐标和标签
class MyDataset(Dataset):
    def __init__(self,tri,cd):
        self.tri=tri
        self.cd=cd
    def __getitem__(self,idx):
        x,y=self.tri[:,idx]
        label=self.cd[x][y]
        return x,y,label
    def __len__(self):
        return self.tri.shape[1]

In [12]:
# argparse.ArgumentParser   是argparse库中的一个类，用于创建命令行参数解析器对象。
    # description   参数用于提供关于这个命令行工具的简短描述，通常会在用户请求帮助信息时显示。
parser = argparse.ArgumentParser(description='PyTorch implementation of ANS-GAT')
#使用argparse库定义了一系列命令行参数，以允许用户通过命令行传递这些参数来自定义脚本的行为。
parser.add_argument('--dataset_name', type=str, default='circ_dis')
parser.add_argument('--num_heads', type=int, default=8)
parser.add_argument('--hidden_dim', type=int, default=1527)
parser.add_argument('--edge_hidden', type=int, default=64)
parser.add_argument('--dropout_rate', type=float, default=0.5)              #模型丢弃率
parser.add_argument('--weight_decay', type=float, default=0.00001)                #权重衰减系数
parser.add_argument('--checkpoint_path', type=str, default='')              #默认为空字符串，表示不保存检查点。
parser.add_argument('--epochs', type=int, default=100)
parser.add_argument('--peak_lr', type=float, default=2e-4)                  #顶峰学习率（最大值）
parser.add_argument('--num_global_node', type=int, default=1)               #全局节点数量
parser.add_argument('--batch_size', type=int, default=32)
#具体的种子数值通常是任意选择的，只要在实验和比较中保持一致即可。
parser.add_argument('--seed', type=int, default=42)                         #随机种子默认42
parser.add_argument('--device', type=int, default=0, help='which gpu to use if any (default: 0)')
                                            #GPU设备编号
parser.add_argument('--weight_update_period', type=int, default=20, help='epochs to update the sampling weight')
                                            #更新采样权重周期数
args = parser.parse_args([])                                            
#解析用户在命令行中传递的参数，并将这些参数的值存储在 args 对象中，以便在脚本的其他部分使用。
#如果CUDA可用且用户指定了GPU设备编号，使用GPU,否则使用CPU计算
device = torch.device("cuda:" + str(args.device)) if torch.cuda.is_available() else torch.device("cpu")
 #加载数据
data_list_cc = torch.load('./data_circ/dataset/data_cc.pt')
data_list_dd = torch.load('./data_circ/dataset/data_dd.pt')
#加载特征
_,cd,_,tri,tei=torch.load('./circ_CNN.pth')     #读取数据
#加载特征
adj = torch.load('./data_circ/dataset/adj')
feature = torch.load('./data_circ/dataset/fea')


 #加载数据
data_list_cc = torch.load('./data_circ/dataset/data_cc.pt')
data_list_dd = torch.load('./data_circ/dataset/data_dd.pt')


#加载特征
adj_cd = torch.load('./data_circ/dataset/adj_cd')
feature_cd = torch.load('./data_circ/dataset/fea_cd')
feature_cd= torch.stack(feature_cd).to(device)
adj_cd= torch.stack(adj_cd).to(device)


data_list_cc = torch.stack([torch.stack(inner_list) for inner_list in data_list_cc]).to(device)
data_list_dd = torch.stack([torch.stack(inner_list) for inner_list in data_list_dd]).to(device)
data_list_cc[data_list_cc == 1527] = 972
data_list_dd[data_list_dd == 1527] = 972

adj_ori = adj_cd[:, :972, :972]     #已经归一化了
feature_ori = feature_cd[:, :972, :]

X_new = torch.matmul(adj_ori, feature_ori)


print('dataset load successfuly')
print(args)
print(feature[0].shape[1])

dataset load successfuly
Namespace(dataset_name='circ_dis', num_heads=8, hidden_dim=1527, edge_hidden=64, dropout_rate=0.5, weight_decay=1e-05, checkpoint_path='', epochs=100, peak_lr=0.0002, num_global_node=1, batch_size=32, seed=42, device=0, weight_update_period=20)
1527


In [4]:
for i in range(5):
    print('cross:%d'%(i+1))
    model = GAT(
        nfeat=1527,
        nhid=args.hidden_dim,
        dropout=args.dropout_rate,
        nheads=args.num_heads,
        alpha=0.2,
    ).to(device)
    trset=DataLoader(MyDataset(tri[i],cd),args.batch_size,shuffle=True)      #读训练数据，格式（32,x1,x2,label）
    teset=DataLoader(MyDataset(tei[i],cd),args.batch_size,shuffle=False)     #读测试数据
    #定义优化器，adamW
    optimizer = torch.optim.Adam(model.parameters(), lr=args.peak_lr,weight_decay=args.weight_decay)
    sampling_weight = np.ones(3)
    ## weight_history 用于记录权重的历史变化
    weight_history = []
    # 设定最小的概率值
    p_min = 0.05
    p = (1 - 3 * p_min) * sampling_weight / sum(sampling_weight) + p_min
    for epoch in range(1, args.epochs+1):
        print("====epoch " + str(epoch))
        #训练
        train(args, model, device, trset, optimizer, feature_cd[i],adj_cd[i],feature_ori[i],adj_ori[i],data_list_cc[i],data_list_dd[i],X_new[i],teset,i,epoch,args.epochs)
        #更新学习调度器参数
#         lr_scheduler.step()
#         检查当前训练周期是否是权重更新的周期
        if epoch % args.weight_update_period == 0:
            #计算验证集上的奖励
            r = get_reward(i,args, model, device, trset, p,feature_cd[i],adj_cd[i],feature_ori[i],adj_ori[i],data_list_cc[i],data_list_dd[i],X_new[i])
            print('reward:', r)
            #更新采样权重=sampling_weight*e^(2r+修正项)
            sampling_weight = sampling_weight * 1 / (1 + np.exp(-r))
            #再将权重转换为概率分布
            p = (1 - 3 * p_min) * sampling_weight / sum(sampling_weight) + p_min
            print('p:', p)
            #保存权重的变化
            weight_history.append(p)
            #根据概率分布 p 进行节点采样，得到新的数据集 data_list 和节点特征 feature
            data_list_cc,data_list_dd = node_sampling(p)
            data_list_cc = torch.stack([torch.stack(inner_list) for inner_list in data_list_cc]).to(device)
            data_list_dd = torch.stack([torch.stack(inner_list) for inner_list in data_list_dd]).to(device)
            data_list_cc[data_list_cc == 1527] = 972
            data_list_dd[data_list_dd == 1527] = 972

cross:1



KeyboardInterrupt

